# MovieLens movie and user vector diagnostics

This notebook visualizes M1, M3, and M4 and checks the original high-dimensional space.

Inspect content groups and differences between user profiles, compare positive and distractor scores, and check feature-block magnitudes.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "data.py").exists():
            return path
    raise FileNotFoundError("Run this notebook inside the dam repository.")

REPO_ROOT = find_repo_root()
PROJECT_DIR = REPO_ROOT / "content_collab" / "movielens_dataset"
for path in (REPO_ROOT, PROJECT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/dam-matplotlib-cache")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import trustworthiness

from movielens_content import (
    build_movie_features,
    build_user_item_weights,
    build_user_profiles,
    prepare_shared_data,
    resolve_dataset_dir,
    score_candidates,
)

SEED = 42
USER_SAMPLE_SIZE = 2_000
QUALITY_SAMPLE_SIZE = 750
DIAGNOSTIC_USERS = 2_000
RANDOM_PAIRS = 20_000

DATASET_DIR = resolve_dataset_dir(
    os.getenv("MOVIELENS_1M_DIR"),
    REPO_ROOT,
    auto_download=True,
)
shared = prepare_shared_data(DATASET_DIR)
features = build_movie_features(shared.metadata)
item_vectors = features.configurations
user_item_weights, user_to_index, item_to_index = build_user_item_weights(shared)
user_vectors = {
    name: build_user_profiles(user_item_weights, matrix)
    for name, matrix in item_vectors.items()
}
print("Movies:", len(shared.warm_items), "Users:", len(shared.eligible_users))

## 1. Projection method

UMAP is used when installed. Otherwise, the notebook uses deterministic Truncated SVD and reports lower-dimensional quality checks.

The original-space cosine diagnostics remain more reliable than visual distance.

In [ ]:
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False

def project(matrix, seed=SEED):
    components = min(50, matrix.shape[0] - 1, matrix.shape[1] - 1)
    if UMAP_AVAILABLE:
        reduced = TruncatedSVD(
            n_components=components,
            random_state=seed,
        ).fit_transform(matrix)
        coordinates = umap.UMAP(
            n_components=2,
            n_neighbors=30,
            min_dist=0.15,
            metric="cosine",
            random_state=seed,
        ).fit_transform(reduced)
        return coordinates, f"SVD-{components} + UMAP"
    return (
        TruncatedSVD(n_components=2, random_state=seed).fit_transform(matrix),
        "Truncated SVD",
    )

def projection_quality(matrix, coordinates):
    rng = np.random.default_rng(SEED)
    indices = rng.choice(
        matrix.shape[0],
        size=min(QUALITY_SAMPLE_SIZE, matrix.shape[0]),
        replace=False,
    )
    original = matrix[indices]
    projected = coordinates[indices]
    score = trustworthiness(
        original.toarray(),
        projected,
        n_neighbors=10,
        metric="cosine",
    )
    original_nn = NearestNeighbors(
        n_neighbors=11,
        metric="cosine",
        algorithm="brute",
    ).fit(original).kneighbors(original, return_distance=False)[:, 1:]
    projected_nn = NearestNeighbors(
        n_neighbors=11,
        metric="euclidean",
    ).fit(projected).kneighbors(projected, return_distance=False)[:, 1:]
    overlap = np.mean([
        len(set(left).intersection(right)) / 10
        for left, right in zip(original_nn, projected_nn)
    ])
    return float(score), float(overlap)

print("UMAP available:", UMAP_AVAILABLE)

## 2. Movie-vector maps

Movies are coloured by their first listed genre.

In [ ]:
def first_genre(value):
    if not value or value == "(no genres listed)":
        return "Missing"
    return value.split("|")[0]

movie_group = shared.metadata["genres"].map(first_genre)
major_groups = set(movie_group.value_counts().head(8).index)
movie_group = movie_group.where(movie_group.isin(major_groups), "Other")
groups = sorted(movie_group.unique())
colours = {group: plt.cm.tab10(i % 10) for i, group in enumerate(groups)}

movie_coordinates = {}
quality_rows = []
for name in ["M1", "M3", "M4"]:
    coordinates, method = project(item_vectors[name])
    movie_coordinates[name] = coordinates
    trust, overlap = projection_quality(item_vectors[name], coordinates)
    quality_rows.append({
        "configuration": name,
        "method": method,
        "trustworthiness@10": trust,
        "top10 neighbour overlap": overlap,
    })
display(pd.DataFrame(quality_rows).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for axis, name in zip(axes, ["M1", "M3", "M4"]):
    coordinates = movie_coordinates[name]
    for group in groups:
        mask = movie_group.eq(group).to_numpy()
        axis.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=10,
            alpha=0.55,
            color=colours[group],
            label=group,
            linewidths=0,
        )
    axis.set_title(f"{name} movie vectors")
    axis.set_xlabel("Projection dimension 1")
    axis.set_ylabel("Projection dimension 2")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, frameon=False)
fig.tight_layout(rect=(0, 0.10, 1, 1))
plt.show()

## 3. User-vector maps

Users are coloured by the strongest genre coordinate in their M1 profile. Marker size represents positive-history size.

In [ ]:
rng = np.random.default_rng(SEED)
sample_indices = np.sort(
    rng.choice(
        len(shared.eligible_users),
        size=min(USER_SAMPLE_SIZE, len(shared.eligible_users)),
        replace=False,
    )
)
genre_names = features.genre_names
sample_m1 = user_vectors["M1"][sample_indices]
dominant_index = np.asarray(sample_m1.argmax(axis=1)).ravel()
user_group = pd.Series(genre_names[dominant_index])
major_user_groups = set(user_group.value_counts().head(8).index)
user_group = user_group.where(user_group.isin(major_user_groups), "Other")
user_groups = sorted(user_group.unique())
user_colours = {
    group: plt.cm.tab10(i % 10) for i, group in enumerate(user_groups)
}
activity = np.asarray(user_item_weights.getnnz(axis=1)).ravel()[sample_indices]
sizes = 8 + 18 * np.log1p(activity) / np.log1p(activity).max()

user_coordinates = {}
user_quality = []
for name in ["M1", "M3", "M4"]:
    matrix = user_vectors[name][sample_indices]
    coordinates, method = project(matrix)
    user_coordinates[name] = coordinates
    trust, overlap = projection_quality(matrix, coordinates)
    user_quality.append({
        "configuration": name,
        "method": method,
        "trustworthiness@10": trust,
        "top10 neighbour overlap": overlap,
    })
display(pd.DataFrame(user_quality).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for axis, name in zip(axes, ["M1", "M3", "M4"]):
    coordinates = user_coordinates[name]
    for group in user_groups:
        mask = user_group.eq(group).to_numpy()
        axis.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=sizes[mask],
            alpha=0.45,
            color=user_colours[group],
            label=group,
            linewidths=0,
        )
    axis.set_title(f"{name} user profiles")
    axis.set_xlabel("Projection dimension 1")
    axis.set_ylabel("Projection dimension 2")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=6, frameon=False)
fig.tight_layout(rect=(0, 0.10, 1, 1))
plt.show()

## 4. Original-space user similarity

High random-pair cosine similarity means many user profiles point in similar directions, reducing personalization.

In [ ]:
pair_rng = np.random.default_rng(SEED)
left = pair_rng.integers(0, len(shared.eligible_users), size=RANDOM_PAIRS)
right = pair_rng.integers(0, len(shared.eligible_users), size=RANDOM_PAIRS)

similarities = {}
rows = []
for name in ["M1", "M3", "M4"]:
    values = np.asarray(
        user_vectors[name][left]
        .multiply(user_vectors[name][right])
        .sum(axis=1)
    ).ravel()
    similarities[name] = values
    rows.append({
        "configuration": name,
        "mean": values.mean(),
        "median": np.median(values),
        "p90": np.quantile(values, 0.90),
    })
display(pd.DataFrame(rows).round(4))

bins = np.linspace(0, 1, 51)
for name, values in similarities.items():
    plt.hist(
        values,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2,
        label=name,
    )
plt.xlabel("Cosine similarity between random user profiles")
plt.ylabel("Density")
plt.title("Original-space user similarity")
plt.legend(frameon=False)
plt.show()

## 5. Hidden-positive versus distractor scores

The first 2,000 sorted users receive the exact prefix of the fixed validation candidate generation.

In [ ]:
diagnostic_users = shared.eligible_users[:DIAGNOSTIC_USERS]
candidates = shared.validation_candidates.loc[
    shared.validation_candidates["user_id"].isin(diagnostic_users)
].reset_index(drop=True)

distribution_rows = []
score_data = {}
for name in ["M1", "M3", "M4"]:
    predictions = score_candidates(
        candidates,
        user_vectors[name],
        item_vectors[name],
        user_to_index,
        item_to_index,
    )
    positive = predictions.loc[predictions["label"].eq(1), "score"].to_numpy()
    distractor = predictions.loc[predictions["label"].eq(0), "score"].to_numpy()
    score_data[name] = (positive, distractor)

    positive_by_user = (
        predictions.loc[predictions["label"].eq(1)]
        .set_index("user_id")["score"]
    )
    negative_rows = predictions.loc[predictions["label"].eq(0)].copy()
    negative_rows["positive_score"] = negative_rows["user_id"].map(
        positive_by_user
    )
    distribution_rows.append({
        "configuration": name,
        "positive median": np.median(positive),
        "distractor median": np.median(distractor),
        "positive>distractor rate": (
            negative_rows["positive_score"] > negative_rows["score"]
        ).mean(),
    })

display(pd.DataFrame(distribution_rows).round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for axis, name in zip(axes, ["M1", "M3", "M4"]):
    positive, distractor = score_data[name]
    combined = np.concatenate([positive, distractor])
    low, high = np.quantile(combined, [0.005, 0.995])
    bins = np.linspace(low, high, 55)
    axis.hist(
        distractor,
        bins=bins,
        density=True,
        alpha=0.55,
        label="Unobserved distractor",
    )
    axis.hist(
        positive,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=2.2,
        label="Hidden positive",
    )
    axis.set_title(name)
    axis.set_xlabel("Cosine score")
    axis.set_ylabel("Density")
axes[-1].legend(frameon=False)
fig.tight_layout()
plt.show()

## 6. Nearest-movie examples

These neighbours are calculated in the original vector spaces.

In [ ]:
anchors = ["Toy Story", "Matrix, The", "Alien", "Godfather, The"]
anchor_rows = {
    title: int(shared.metadata.index[shared.metadata["title"].eq(title)][0])
    for title in anchors
    if shared.metadata["title"].eq(title).any()
}
neighbour_rows = []
for name in ["M1", "M3", "M4"]:
    model = NearestNeighbors(
        n_neighbors=6,
        metric="cosine",
        algorithm="brute",
    ).fit(item_vectors[name])
    for title, row in anchor_rows.items():
        distances, indices = model.kneighbors(item_vectors[name][row])
        for rank, (distance, index) in enumerate(
            zip(distances[0][1:], indices[0][1:]),
            start=1,
        ):
            neighbour_rows.append({
                "configuration": name,
                "anchor": title,
                "rank": rank,
                "neighbour": shared.metadata.iloc[index]["title_with_year"],
                "similarity": 1 - float(distance),
                "genres": shared.metadata.iloc[index]["genres"],
            })
display(pd.DataFrame(neighbour_rows).round({"similarity": 4}))

## 7. Feature-block balance

Because every block is normalized before concatenation, no large vocabulary automatically dominates the vector.

In [ ]:
def energy(matrix):
    return np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel()

energies = {
    "genre": energy(features.genre_block),
    "title": energy(features.title_block),
    "decade": energy(features.decade_block),
}
balance_rows = []
for configuration, blocks in {
    "M2": ["genre", "decade"],
    "M3": ["genre", "title"],
    "M4": ["genre", "title", "decade"],
}.items():
    total = sum(energies[block] for block in blocks)
    for block in blocks:
        share = np.divide(
            energies[block],
            total,
            out=np.zeros_like(total),
            where=total > 0,
        )
        balance_rows.append({
            "configuration": configuration,
            "block": block,
            "mean share": share.mean(),
            "median share": np.median(share),
        })
balance = pd.DataFrame(balance_rows)
display(balance.round(4))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, configuration in zip(axes, ["M2", "M3", "M4"]):
    current = balance.loc[balance["configuration"].eq(configuration)]
    axis.bar(current["block"], current["median share"])
    axis.set_ylim(0, 1)
    axis.set_title(configuration)
    axis.set_ylabel("Median squared-magnitude share")
fig.tight_layout()
plt.show()

## Reading the diagnostics

- Are movie clusters understandable from genres, titles, or decades?
- Are user profiles distinct or concentrated?
- Do hidden positives shift to higher cosine scores?
- Does M3/M4 retrieve franchises rather than semantic similarity?
- Do the original-space diagnostics agree with the projected maps?
- Are low projection-quality maps being interpreted cautiously?